# Training Notebook (One time)

In [1]:
import kagglehub
import shutil
import os

# Download to cache
cache_path = kagglehub.dataset_download('mehmoodsheikh/fairface-dataset')
source_dir = os.path.join(cache_path, 'FairFace', 'fairface-img-margin025-trainval')

# Target local directory
local_dataset_path = 'fairface-dataset'

# Copy if it doesn't exist locally
if not os.path.exists(local_dataset_path):
    print(f"Copying dataset from {source_dir} to {local_dataset_path}...")
    shutil.copytree(source_dir, local_dataset_path)
    print("Copy complete.")
else:
    print(f"Dataset already exists at {local_dataset_path}")

DATA_ROOT = f'{local_dataset_path}'
TRAIN_CSV = f'{local_dataset_path}/fairface_label_train.csv'
VAL_CSV = f'{local_dataset_path}/fairface_label_val.csv'

f:\Documents\vgg16\ai331\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset already exists at fairface-dataset


In [ ]:
import os
import csv
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
import torch.optim as optim

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class FairFaceDataset(Dataset):
    def __init__(self, csv_path, img_root, transform=None):
        self.img_root = img_root
        self.transform = transform
        self.samples = []
        with open(csv_path, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.samples.append(row)
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        row = self.samples[idx]
        img_path = os.path.join(self.img_root, row['file'])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        gender = 1.0 if row['gender'] == 'Female' else 0.0
        age_map = {
            # Fine-grained bins
            '0-4': 2.5, '5-9': 7.0, '10-14': 12.0, '15-19': 17.0,
            '20-24': 22.0, '25-29': 27.0, '30-34': 32.0, '35-39': 37.0,
            '40-44': 42.0, '45-49': 47.0, '50-54': 52.0, '55-59': 57.0,
            '60-64': 62.0, '65-69': 67.0, '70+': 75.0,
            # Coarse bins (common FairFace format)
            '0-2': 1.0, '3-9': 6.0, '10-19': 14.5, '20-29': 24.5,
            '30-39': 34.5, '40-49': 44.5, '50-59': 54.5, '60-69': 64.5,
            'more than 70': 75.0
        }

        age_label = row['age'].strip()
        if age_label in age_map:
            age_value = age_map[age_label]
        elif '-' in age_label:
            start, end = age_label.split('-', 1)
            age_value = (float(start) + float(end)) / 2.0
        elif age_label.endswith('+'):
            age_value = float(age_label[:-1]) + 5.0
        else:
            age_value = 37.5  # safe fallback to avoid KeyError

        age = age_value / 75.0
        race_map = { 'East Asian': 0, 'Indian': 1, 'Black': 2, 'White': 3, 'Middle Eastern': 4, 'Latino_Hispanic': 5, 'Southeast Asian': 6 }
        race = race_map.get(row['race'].strip(), 0)
        return image, torch.tensor(gender), torch.tensor(age), torch.tensor(race, dtype=torch.long)

if os.path.exists(TRAIN_CSV):
    ds = FairFaceDataset(TRAIN_CSV, DATA_ROOT, train_transform)
    print('Samples:', len(ds))
    img, gender, age, race = ds[0]
    print('Image shape:', img.shape, 'Gender:', gender.item(), 'Age:', age.item(), 'Race:', race.item())
else:
    print('CSV not found. Please check paths.')

Samples: 86744
Image shape: torch.Size([3, 224, 224]) Gender: 0.0 Age: 0.7266666889190674 Race: 0


In [ ]:
BATCH_SIZE = 32
if os.path.exists(TRAIN_CSV):
    train_ds = FairFaceDataset(TRAIN_CSV, DATA_ROOT, train_transform)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    print('Batch:', next(iter(train_loader))[0].shape)
else:
    train_loader = None

Batch: torch.Size([32, 3, 224, 224])


In [ ]:
# Simple VGG16 multi-task model
class VGG16Multi(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        for param in vgg.parameters():
            param.requires_grad = False
        self.features = vgg.features
        self.avgpool = vgg.avgpool
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(512*7*7, 512), nn.ReLU(), nn.Dropout(0.5)
        )
        self.gender = nn.Sequential(nn.Linear(512, 1), nn.Sigmoid())
        self.age = nn.Sequential(nn.Linear(512, 1), nn.Sigmoid())
        self.race = nn.Linear(512, 7) # 7 race classes
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        x = self.fc(x)
        gender = self.gender(x)
        age = self.age(x)
        race = self.race(x)
        return gender, age, race

model = VGG16Multi()
print(model)

VGG16Multi(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dil

In [ ]:
# Training loop with accuracy visualization
import matplotlib.pyplot as plt

if train_loader:
    # Validation loader
    if os.path.exists(VAL_CSV):
        val_ds = FairFaceDataset(VAL_CSV, DATA_ROOT, val_transform)
        val_loader = DataLoader(val_ds, batch_size=32)
    else:
        val_loader = None

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    model = model.to(device)
    criterion_gender = nn.BCELoss()
    criterion_age = nn.L1Loss()
    criterion_race = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
    num_epochs = 15
    train_accs, val_accs = [], []
    for epoch in range(num_epochs):
        model.train()
        correct, total = 0, 0
        for images, gender, age, race in train_loader:
            images = images.to(device)
            gender = gender.to(device).unsqueeze(1)
            age = age.to(device).unsqueeze(1)
            race = race.to(device)
            optimizer.zero_grad()
            out_gender, out_age, out_race = model(images)
            loss_gender = criterion_gender(out_gender, gender)
            loss_age = criterion_age(out_age, age)
            loss_race = criterion_race(out_race, race)
            loss = loss_gender + loss_age + loss_race
            loss.backward()
            optimizer.step()
            # Gender accuracy
            preds = (out_gender > 0.5).float()
            correct += (preds == gender).sum().item()
            total += gender.size(0)
        train_acc = correct / total if total > 0 else 0
        train_accs.append(train_acc)
        # Validation accuracy
        if val_loader:
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for images, gender, age, race in val_loader:
                    images = images.to(device)
                    gender = gender.to(device).unsqueeze(1)
                    out_gender, _, _ = model(images)
                    preds = (out_gender > 0.5).float()
                    correct += (preds == gender).sum().item()
                    total += gender.size(0)
            val_acc = correct / total if total > 0 else 0
            val_accs.append(val_acc)
        else:
            val_accs.append(None)
        print(f"Epoch {epoch+1}/{num_epochs} - Train Acc: {train_acc:.4f} - Val Acc: {val_acc if val_loader else 'N/A'}")
    torch.save(model.state_dict(), 'model.pth')
    print('Model saved to model.pth')
    # Plot
    plt.figure(figsize=(8,5))
    plt.plot(train_accs, color='green', label='Train Accuracy')
    if any(val_accs):
        plt.plot(val_accs, color='blue', label='Val Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Gender Accuracy')
    plt.legend()
    plt.title('Training & Validation Accuracy')
    plt.show()
else:
    print('No training data loaded.')

Using device: cuda
Epoch 1/15 - Train Acc: 0.7657 - Val Acc: 0.8132189154646704
